# Session 13 · Case study: vocabulary and document-term matrix

**Block 1 practice.** Build the vocabulary of the reviews and inspect the document-term matrix.

You will

1. count tokens and types in the 50,000-review sample,
2. build a document-term matrix with `CountVectorizer` and measure its size and sparsity,
3. look at the most frequent and the rarest words (Zipf's law),
4. measure how preprocessing changes the vocabulary.

Theory: [01-text-as-data.md](../theory/01-text-as-data.md). Data: `case-study/data/train_sample.parquet` (prepare it with `case-study/prepare_data.py`).

*Author: course team, licence CC-BY-4.0.*

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").exists())
DATA = ROOT / "case-study" / "data"
reviews = pd.read_parquet(DATA / "train_sample.parquet")
texts = reviews["title"] + " " + reviews["text"]          # one document = title + text
y = reviews["label"]
print(reviews.shape, y.value_counts(normalize=True).round(3).to_dict())

## 1. Tokens and types

A first, naive tokenisation: lower-case and split at white space.

In [ ]:
tokens = texts.str.lower().str.split()
n_tokens = tokens.str.len()
print("tokens in total:", int(n_tokens.sum()))
print(n_tokens.describe().round(1))

types = set()
for t in tokens:
    types.update(t)
print("types (distinct tokens):", len(types))

**Question.** Why is the naive number of types much larger than the vocabulary that `CountVectorizer` finds below? Look at a few types that contain punctuation.

In [ ]:
print(sorted(t for t in types if t.startswith("great"))[:15])

## 2. The document-term matrix

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()                       # lower-case, default token pattern
D = cv.fit_transform(texts)                  # scipy sparse CSR matrix
vocab = cv.get_feature_names_out()
print("shape:", D.shape)
print("non-zero cells:", D.nnz, f"({D.nnz / (D.shape[0] * D.shape[1]):.5f} of all cells)")
print("distinct words per review (mean):", round(D.nnz / D.shape[0], 1))
sparse_mb = (D.data.nbytes + D.indices.nbytes + D.indptr.nbytes) / 1e6
print(f"memory: {sparse_mb:.1f} MB sparse vs {D.shape[0] * D.shape[1] * 8 / 1e9:.1f} GB dense")

Inspect one review and its row of the matrix. Only the non-zero entries are stored.

In [ ]:
i = 7
print(texts.iloc[i][:300])
row = D[i]
print(pd.Series(row.data, index=vocab[row.indices]).sort_values(ascending=False).head(15))

## 3. Frequent and rare words (Zipf's law)

In [ ]:
import matplotlib.pyplot as plt

counts = pd.Series(np.asarray(D.sum(axis=0)).ravel(), index=vocab).sort_values(ascending=False)
doc_freq = pd.Series(np.bincount(D.indices, minlength=D.shape[1]), index=vocab)
print("most frequent:", counts.head(10).index.tolist())
print("share of words that occur only once:", round((counts == 1).mean(), 2))
print("words in at least 5 reviews:", int((doc_freq >= 5).sum()))

fig, ax = plt.subplots(figsize=(6, 4))
ax.loglog(np.arange(1, len(counts) + 1), counts.to_numpy())
ax.set_xlabel("rank of word (log)")
ax.set_ylabel("frequency (log)")
ax.set_title("Zipf's law in the review sample")
plt.show()

**Question.** The plot is close to a straight line on log–log axes. What does that mean for the size of the vocabulary if we used all 434,373 training reviews instead of 50,000?

## 4. Preprocessing and the size of the vocabulary

In [ ]:
from nltk.stem import PorterStemmer

stemmer = PorterStemmer()
base = CountVectorizer().build_analyzer()          # the default tokeniser as a function


def stem_analyzer(doc):
    return [stemmer.stem(t) for t in base(doc)]


settings = {
    "default (lower-case)": CountVectorizer(),
    "no lower-casing": CountVectorizer(lowercase=False),
    "English stop words removed": CountVectorizer(stop_words="english"),
    "min_df=5": CountVectorizer(min_df=5),
    "Porter stemming": CountVectorizer(analyzer=stem_analyzer),
    "unigrams + bigrams, min_df=3": CountVectorizer(ngram_range=(1, 2), min_df=3),
}
sample = texts.sample(10_000, random_state=0)      # a subsample keeps stemming fast
rows = []
for name, vec in settings.items():
    M = vec.fit_transform(sample)
    rows.append({"setting": name, "vocabulary": M.shape[1], "non-zeros": M.nnz})
pd.DataFrame(rows)

## Your turn

1. Which setting removes the most columns? Which one could hurt a sentiment classifier, and why?
2. Find three words in the vocabulary that a lemmatiser would merge but the default tokeniser keeps apart (hint: search `vocab` for `"batter"` or `"work"`).
3. Compute the share of reviews that contain the word `not` (use `doc_freq`). Would you remove it?
4. Optional: compare the vocabulary of negative and positive reviews: which words occur in at least 1 % of negative reviews but in fewer than 0.2 % of positive ones?

In [ ]:
# your code here
print([w for w in vocab if w.startswith("batter")])